In [ ]:
import os
from dotenv import load_dotenv

from lib.agentic_rag import AgenticRAG, KeywordSearchTool, SEARCH_TOOL_DEFINITION

from openai import OpenAI
from sqlitesearch import TextSearchIndex
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
load_dotenv()

In [ ]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini"))
)

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

In [ ]:
result.cost

In [ ]:
result = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

In [ ]:
runner.run()